# Tutorial 2-2: Data Transformation
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

## Objective
Now that our data is clean, we must prepare it for the mathematical requirements of machine learning algorithms. In this session, we will explore:
1. **Aggregation:** Reducing data volume and stabilizing variability.
2. **Sampling:** Selecting a representative subset that maintains the properties of the original population.
3. **Discretization:** Converting continuous attributes into categorical buckets (Equal Width vs. Equal Frequency).
4. **Attribute Transformation:** Scaling features via Min-Max Normalization and Z-Score Standardization.

## 1. Aggregation and Scale
Aggregation is the process of combining multiple objects into a single summary. This is often used for data reduction or to change the scale of analysis (e.g., converting hourly sensor data into daily averages). 

**Why do this?** Aggregated data tends to be more 'stable'—it smooths out random noise and high-frequency fluctuations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# Load housing data
housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
df_housing['TargetValue'] = housing.target

# Let's aggregate by 'HouseAge' to see the average 'MedInc' (Median Income) for different age groups
age_aggregated = df_housing.groupby('HouseAge')['MedInc'].mean().reset_index()

plt.figure(figsize=(10, 5))
plt.plot(df_housing['HouseAge'][:100], df_housing['MedInc'][:100], alpha=0.3, label='Raw Data (first 100)')
plt.plot(age_aggregated['HouseAge'], age_aggregated['MedInc'], color='red', label='Aggregated Mean')
plt.title("Aggregation: Smoothing Variability")
plt.xlabel("House Age")
plt.ylabel("Median Income")
plt.legend()
plt.show()

## 2. Sampling Techniques
The key principle of sampling is that a sample works as well as the entire dataset *if it is representative*. We will compare **Simple Random Sampling** with **Stratified Sampling**.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Simple Random Sampling
random_sample = df_housing.sample(frac=0.1, random_state=42)

# 2. Stratified Sampling
# We use this when we want to ensure specific groups are represented proportionally.
# Let's create a dummy category for stratification
df_housing['Income_Cat'] = pd.cut(df_housing['MedInc'], bins=5, labels=False)

strat_sample, _ = train_test_split(df_housing, test_size=0.9, stratify=df_housing['Income_Cat'], random_state=42)

print(f"Original Mean Income: {df_housing['MedInc'].mean():.4f}")
print(f"Random Sample Mean: {random_sample['MedInc'].mean():.4f}")
print(f"Stratified Sample Mean: {strat_sample['MedInc'].mean():.4f}")

## 3. Discretization
Discretization is the process of turning a continuous attribute into an ordinal one. This is essential for algorithms that cannot handle continuous values or to simplify complex relationships.

### 3.1 Equal Width vs. Equal Frequency
- **Equal Width:** Divides the range of values into $N$ intervals of the same size. Highly sensitive to outliers.
- **Equal Frequency (Quantile):** Divides the data so that each interval contains approximately the same number of points.

In [ ]:
feature = df_housing['MedInc']

# Equal Width (3 bins: Low, Medium, High)
width_bins = pd.cut(feature, bins=3)

# Equal Frequency (3 bins: 33rd, 66th percentiles)
freq_bins = pd.qcut(feature, q=3)

print("Equal Width Bin Counts:")
print(width_bins.value_counts())
print("\nEqual Frequency Bin Counts:")
print(freq_bins.value_counts())

## 4. Attribute Transformation (Scaling)
Most machine learning models (like K-NN or Neural Networks) calculate distances between points. If one attribute ranges from 0-1 and another from 0-1,000,000, the larger attribute will dominate the calculations. 

### 4.1 Min-Max Normalization
Maps all values to the range [0, 1]. 
Formula: $x' = (x - min) / (max - min)$

### 4.2 Z-Score Standardization
Transforms data to have a mean of 0 and a standard deviation of 1. 
Formula: $x' = (x - \mu) / \sigma$

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

data_to_scale = df_housing[['MedInc', 'HouseAge']]

# Min-Max Scaling
mm_scaler = MinMaxScaler()
minmax_scaled = mm_scaler.fit_transform(data_to_scale)

# Standardization
std_scaler = StandardScaler()
zscore_scaled = std_scaler.fit_transform(data_to_scale)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].scatter(data_to_scale.iloc[:, 0], data_to_scale.iloc[:, 1], alpha=0.5)
ax[0].set_title("Original Data")

ax[1].scatter(minmax_scaled[:, 0], minmax_scaled[:, 1], color='green', alpha=0.5)
ax[1].set_title("Min-Max Scaled [0, 1]")

ax[2].scatter(zscore_scaled[:, 0], zscore_scaled[:, 1], color='red', alpha=0.5)
ax[2].set_title("Z-Score Standardized (Mean 0)")

plt.show()

## Summary
Transformation is about making data 'algorithm-friendly'. We learned how to:
1. Aggregate data to stabilize variability.
2. Use stratified sampling to maintain population proportions.
3. Bin continuous data using width and frequency methods.
4. Scale data so that no single attribute unfairly dominates the model.

Next, we will apply these logic-based transformations to the specialized world of **Text Data**.